In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

torch_device = torch.device(device)
print({'selected_device': device})


In [ ]:
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

goemotions = load_dataset('go_emotions', 'raw', split='test')

goemotions_to_canonical = {
    'sadness': 'sadness',
    'grief': 'sadness',
    'disappointment': 'sadness',
    'remorse': 'sadness',
    'embarrassment': 'sadness',
    'joy': 'joy',
    'amusement': 'joy',
    'excitement': 'joy',
    'optimism': 'joy',
    'pride': 'joy',
    'relief': 'joy',
    'gratitude': 'joy',
    'approval': 'joy',
    'admiration': 'joy',
    'caring': 'love',
    'desire': 'love',
    'love': 'love',
    'anger': 'anger',
    'annoyance': 'anger',
    'disapproval': 'anger',
    'disgust': 'anger',
    'fear': 'fear',
    'nervousness': 'fear',
    'surprise': 'surprise',
    'realization': 'surprise',
    'confusion': 'surprise'
}

selected_rows = []
for row in goemotions:
    labels = row['labels']
    if len(labels) != 1:
        continue
    raw_label = goemotions.features['labels'].feature.int2str(labels[0])
    if raw_label in goemotions_to_canonical:
        selected_rows.append({
            'text': row['text'],
            'goemotions_label': raw_label,
            'canonical_label': goemotions_to_canonical[raw_label]
        })
    if len(selected_rows) >= 300:
        break

subset_df = pd.DataFrame(selected_rows)
label_to_id = {name: i for i, name in enumerate(class_names)}
true_ids = [label_to_id[x] for x in subset_df['canonical_label']]

print({
    'dataset': 'go_emotions',
    'config': 'raw',
    'split': 'test',
    'subset_size': len(subset_df),
    'class_distribution': subset_df['canonical_label'].value_counts().to_dict()
})
print(subset_df.head(10).to_dict(orient='records'))


In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(torch_device)
model.eval()

label_prompts = [
    'This text expresses sadness.',
    'This text expresses joy.',
    'This text expresses love.',
    'This text expresses anger.',
    'This text expresses fear.',
    'This text expresses surprise.'
]

print({
    'model_name': model_name,
    'task': 'embedding-similarity-classification',
    'candidate_labels': class_names,
    'label_prompts': label_prompts,
    'device': device
})


In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=32, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded = {k: v.to(torch_device) for k, v in encoded.items()}
            outputs = model(**encoded)
            embeddings = mean_pool(outputs.last_hidden_state, encoded['attention_mask'])
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)


In [ ]:
texts = subset_df['text'].tolist()

text_embeddings = encode_texts(texts, batch_size=32, max_length=128)
label_embeddings = encode_texts(label_prompts, batch_size=16, max_length=32)

similarity_matrix = text_embeddings @ label_embeddings.T
pred_ids = similarity_matrix.argmax(dim=1).numpy().tolist()
pred_labels = [class_names[i] for i in pred_ids]
pred_scores = similarity_matrix.max(dim=1).values.numpy().tolist()

results_df = subset_df.copy()
results_df['predicted_label'] = pred_labels
results_df['similarity_score'] = pred_scores

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'go_emotions',
    'config': 'raw',
    'split': 'test',
    'subset_size': len(results_df),
    'device': device,
    'accuracy': round(float(accuracy), 6)
})
print(report)


In [ ]:
sample_n = 12
print(results_df[['text', 'goemotions_label', 'canonical_label', 'predicted_label', 'similarity_score']].head(sample_n).to_string(index=False))
